In [1]:
# Upload train.csv file
from google.colab import files
uploaded = files.upload()

Saving test.csv to test.csv
Saving train.csv to train.csv


In [2]:
# ============================================
# STEP 1: IMPORT LIBRARIES
# ============================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import pickle
import warnings
warnings.filterwarnings('ignore')

print("=" * 80)
print("       🤖 AI MOBILE PRICE PREDICTOR")
print("    Machine Learning Project")
print("    By: Jiten Hudda")
print("=" * 80)
print("\n✅ All libraries imported successfully!")


       🤖 AI MOBILE PRICE PREDICTOR
    Machine Learning Project
    By: Jiten Hudda

✅ All libraries imported successfully!


In [3]:
# ============================================
# STEP 2: LOAD AND EXPLORE DATASET
# ============================================

print("\n📂 LOADING DATASET...")
print("-" * 80)

# Load the CSV file
train_df = pd.read_csv('train.csv')

print(f"✅ Dataset loaded successfully!")
print(f"\n📊 Dataset Information:")
print(f"   • Total samples: {len(train_df)}")
print(f"   • Total features: {len(train_df.columns)-1}")
print(f"   • Target variable: price_range (0-3)")

print(f"\n📈 Price Range Distribution:")
for i in range(4):
    count = len(train_df[train_df['price_range'] == i])
    print(f"   • Class {i}: {count} samples (25.0%)")

print("\n📋 First 5 rows of dataset:")
print(train_df.head())

print("\n📋 Column names:")
print(train_df.columns.tolist())



📂 LOADING DATASET...
--------------------------------------------------------------------------------
✅ Dataset loaded successfully!

📊 Dataset Information:
   • Total samples: 2000
   • Total features: 20
   • Target variable: price_range (0-3)

📈 Price Range Distribution:
   • Class 0: 500 samples (25.0%)
   • Class 1: 500 samples (25.0%)
   • Class 2: 500 samples (25.0%)
   • Class 3: 500 samples (25.0%)

📋 First 5 rows of dataset:
   battery_power  blue  clock_speed  dual_sim  fc  four_g  int_memory  m_dep  \
0            842     0          2.2         0   1       0           7    0.6   
1           1021     1          0.5         1   0       1          53    0.7   
2            563     1          0.5         1   2       1          41    0.9   
3            615     1          2.5         0   0       0          10    0.8   
4           1821     1          1.2         0  13       1          44    0.6   

   mobile_wt  n_cores  ...  px_height  px_width   ram  sc_h  sc_w  talk_time  \

In [4]:
# ============================================
# STEP 3: DATA PREPARATION
# ============================================

print("\n🔍 PREPARING DATA FOR TRAINING...")
print("-" * 80)

# Separate features (X) and target (y)
X = train_df.drop('price_range', axis=1)
y = train_df['price_range']

print(f"✅ Features separated:")
print(f"   • X (features) shape: {X.shape}")
print(f"   • y (target) shape: {y.shape}")

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% for testing
    random_state=42,    # For reproducibility
    stratify=y          # Equal distribution of classes
)

print(f"\n✅ Data split completed:")
print(f"   • Training set: {len(X_train)} samples ({len(X_train)/len(X)*100:.1f}%)")
print(f"   • Testing set: {len(X_test)} samples ({len(X_test)/len(X)*100:.1f}%)")



🔍 PREPARING DATA FOR TRAINING...
--------------------------------------------------------------------------------
✅ Features separated:
   • X (features) shape: (2000, 20)
   • y (target) shape: (2000,)

✅ Data split completed:
   • Training set: 1600 samples (80.0%)
   • Testing set: 400 samples (20.0%)


In [5]:
# ============================================
# STEP 4: TRAIN ML MODEL
# ============================================

print("\n🤖 TRAINING RANDOM FOREST MODEL...")
print("-" * 80)

print("📚 Algorithm: Random Forest Classifier")
print("📝 Hyperparameters:")
print("   • n_estimators = 100 (number of trees)")
print("   • max_depth = 20 (maximum tree depth)")
print("   • random_state = 42 (for reproducibility)")

# Create the model
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    random_state=42,
    n_jobs=-1  # Use all CPU cores
)

# Train the model
print("\n⏳ Training in progress...")
model.fit(X_train, y_train)

print("✅ Model trained successfully!")

# Display feature importance
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False)

print("\n⭐ Top 5 Most Important Features:")
for idx, row in feature_importance.head(5).iterrows():
    bar = '█' * int(row['Importance'] * 100)
    print(f"   {row['Feature']:20s}: {row['Importance']:.4f} {bar}")



🤖 TRAINING RANDOM FOREST MODEL...
--------------------------------------------------------------------------------
📚 Algorithm: Random Forest Classifier
📝 Hyperparameters:
   • n_estimators = 100 (number of trees)
   • max_depth = 20 (maximum tree depth)
   • random_state = 42 (for reproducibility)

⏳ Training in progress...
✅ Model trained successfully!

⭐ Top 5 Most Important Features:
   ram                 : 0.4810 ████████████████████████████████████████████████
   battery_power       : 0.0731 ███████
   px_width            : 0.0562 █████
   px_height           : 0.0559 █████
   mobile_wt           : 0.0388 ███


In [6]:
# ============================================
# STEP 5: MODEL EVALUATION
# ============================================

print("\n📊 EVALUATING MODEL PERFORMANCE...")
print("-" * 80)

# Make predictions
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

# Calculate accuracies
train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)

print(f"\n🎯 Model Accuracy:")
print(f"   • Training Accuracy: {train_acc*100:.2f}%")
print(f"   • Testing Accuracy: {test_acc*100:.2f}%")
print(f"   • Overfitting: {(train_acc - test_acc)*100:.2f}%")

# Detailed classification report
print("\n📋 Detailed Classification Report:")
print(classification_report(y_test, y_test_pred,
                          target_names=['Budget (0)', 'Mid-Range (1)',
                                       'Premium (2)', 'Flagship (3)']))

# Save the model
print("\n💾 Saving model...")
with open('price_model.pkl', 'wb') as f:
    pickle.dump(model, f)
print("✅ Model saved as 'price_model.pkl'")

print("\n" + "=" * 80)
print("✅ MODEL TRAINING COMPLETED!")
print("=" * 80)



📊 EVALUATING MODEL PERFORMANCE...
--------------------------------------------------------------------------------

🎯 Model Accuracy:
   • Training Accuracy: 100.00%
   • Testing Accuracy: 88.25%
   • Overfitting: 11.75%

📋 Detailed Classification Report:
               precision    recall  f1-score   support

   Budget (0)       0.95      0.96      0.96       100
Mid-Range (1)       0.83      0.85      0.84       100
  Premium (2)       0.83      0.78      0.80       100
 Flagship (3)       0.92      0.94      0.93       100

     accuracy                           0.88       400
    macro avg       0.88      0.88      0.88       400
 weighted avg       0.88      0.88      0.88       400


💾 Saving model...
✅ Model saved as 'price_model.pkl'

✅ MODEL TRAINING COMPLETED!


In [7]:
# ============================================
# STEP 6: INTERACTIVE WEB INTERFACE
# ============================================

from IPython.display import HTML, display

html_code = f"""
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>AI-Mobile Price Predictor</title>
    <style>
        * {{
            margin: 0;
            padding: 0;
            box-sizing: border-box;
        }}

        body {{
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            padding: 30px 20px;
            min-height: 100vh;
        }}

        .container {{
            max-width: 650px;
            margin: 0 auto;
        }}

        h1 {{
            color: #ffffff;
            font-size: 2.8rem;
            text-align: center;
            margin-bottom: 35px;
            text-shadow: 3px 3px 6px rgba(0,0,0,0.3);
            font-weight: 700;
        }}

        .card {{
            background: white;
            border-radius: 16px;
            padding: 35px;
            box-shadow: 0 15px 50px rgba(0,0,0,0.25);
        }}

        .card h2 {{
            color: #2c3e50;
            font-size: 1.8rem;
            margin-bottom: 30px;
            padding-bottom: 15px;
            border-bottom: 3px solid #667eea;
            font-weight: 600;
        }}

        .form-group {{
            margin-bottom: 22px;
        }}

        .form-group label {{
            display: block;
            color: #2c3e50;
            font-weight: 700;
            font-size: 1.05rem;
            margin-bottom: 10px;
            letter-spacing: 0.3px;
        }}

        .form-group input,
        .form-group select {{
            width: 100%;
            padding: 15px 18px;
            border: 2.5px solid #d0d0d0;
            border-radius: 10px;
            font-size: 1.08rem;
            font-weight: 500;
            transition: all 0.3s ease;
            background: white;
            color: #2c3e50;
        }}

        .form-group input:focus,
        .form-group select:focus {{
            outline: none;
            border-color: #667eea;
            background: #f7f9ff;
            box-shadow: 0 0 0 4px rgba(102, 126, 234, 0.12);
            transform: translateY(-1px);
        }}

        .form-group select {{
            cursor: pointer;
            appearance: none;
            background-image: url("data:image/svg+xml,%3Csvg xmlns='http://www.w3.org/2000/svg' width='14' height='14' viewBox='0 0 12 12'%3E%3Cpath fill='%23667eea' d='M6 9L1 4h10z'/%3E%3C/svg%3E");
            background-repeat: no-repeat;
            background-position: right 18px center;
            padding-right: 45px;
        }}

        .form-group select option {{
            padding: 12px;
            font-size: 1.05rem;
            font-weight: 500;
        }}

        .btn {{
            width: 100%;
            padding: 18px;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white;
            border: none;
            border-radius: 10px;
            font-size: 1.2rem;
            font-weight: 700;
            cursor: pointer;
            margin-top: 20px;
            transition: all 0.3s ease;
            box-shadow: 0 6px 20px rgba(102, 126, 234, 0.4);
            letter-spacing: 0.5px;
        }}

        .btn:hover {{
            background: linear-gradient(135deg, #5568d3 0%, #6a3d8f 100%);
            transform: translateY(-3px);
            box-shadow: 0 10px 30px rgba(102, 126, 234, 0.5);
        }}

        .btn:active {{
            transform: translateY(-1px);
        }}

        .result {{
            margin-top: 30px;
            padding: 30px;
            background: linear-gradient(135deg, #11998e 0%, #38ef7d 100%);
            border-radius: 12px;
            text-align: center;
            display: none;
            animation: slideIn 0.5s ease;
            box-shadow: 0 8px 25px rgba(17, 153, 142, 0.3);
        }}

        @keyframes slideIn {{
            from {{
                opacity: 0;
                transform: translateY(30px);
            }}
            to {{
                opacity: 1;
                transform: translateY(0);
            }}
        }}

        .result.show {{
            display: block;
        }}

        .result h3 {{
            color: white;
            font-size: 1.3rem;
            margin-bottom: 15px;
            font-weight: 600;
        }}

        .result .price {{
            color: white;
            font-size: 3rem;
            font-weight: 800;
            margin: 15px 0;
            text-shadow: 2px 2px 4px rgba(0,0,0,0.2);
        }}

        .result .category {{
            color: white;
            font-size: 1.3rem;
            font-weight: 600;
            opacity: 0.95;
        }}

        .stats {{
            margin-top: 30px;
            padding: 22px;
            background: linear-gradient(135deg, #f8f9fa 0%, #e9ecef 100%);
            border-radius: 10px;
            text-align: center;
            border: 2px solid #dee2e6;
        }}

        .stats p {{
            color: #495057;
            font-size: 1rem;
            margin: 8px 0;
            font-weight: 500;
        }}

        .stats strong {{
            color: #667eea;
            font-weight: 700;
        }}

        .footer {{
            text-align: center;
            color: white;
            margin-top: 30px;
            font-size: 1rem;
            text-shadow: 1px 1px 3px rgba(0,0,0,0.3);
        }}

        .footer p {{
            margin: 8px 0;
        }}
    </style>
</head>
<body>
    <div class="container">
        <h1>🤖 AI Mobile Price Predictor</h1>

        <div class="card">
            <h2>📱 Price Prediction</h2>

            <form id="priceForm">
                <div class="form-group">
                    <label for="original_price">💰 Original Price (rupees)</label>
                    <input type="number" id="original_price" value="25000" min="5000" max="150000" step="1000" required>
                </div>

                <div class="form-group">
                    <label for="product_age">📅 Product Age (years)</label>
                    <input type="number" id="product_age" value="1" min="0" max="5" step="0.5" required>
                </div>

                <div class="form-group">
                    <label for="condition">⚙️ Phone Condition</label>
                    <select id="condition" required>
                        <option value="5">⭐⭐⭐⭐⭐ Excellent (Like New)</option>
                        <option value="4" selected>⭐⭐⭐⭐ Very Good (Minor Scratches)</option>
                        <option value="3">⭐⭐⭐ Good (Normal Wear)</option>
                        <option value="2">⭐⭐ Fair (Visible Damage)</option>
                        <option value="1">⭐ Poor (Heavy Damage)</option>
                    </select>
                </div>

                <div class="form-group">
                    <label for="battery">🔋 Battery Capacity (mAh)</label>
                    <select id="battery" required>
                        <option value="3000">🔋 3000 mAh (Basic)</option>
                        <option value="4000">🔋 4000 mAh (Standard)</option>
                        <option value="5000" selected>🔋 5000 mAh (Good)</option>
                        <option value="6000">🔋 6000 mAh (Excellent)</option>
                    </select>
                </div>

                <div class="form-group">
                    <label for="storage">💾 Storage Capacity</label>
                    <select id="storage" required>
                        <option value="64">💾 64 GB</option>
                        <option value="128" selected>💾 128 GB</option>
                        <option value="256">💾 256 GB</option>
                        <option value="512">💾 512 GB</option>
                        <option value="1024">💾 1 TB</option>
                    </select>
                </div>

                <div class="form-group">
                    <label for="network">📡 Network Support</label>
                    <select id="network" required>
                        <option value="4g">📡 4G Only</option>
                        <option value="5g" selected>📡 5G Supported</option>
                    </select>
                </div>

                <div class="form-group">
                    <label for="dual_sim">📞 Dual SIM Support</label>
                    <select id="dual_sim" required>
                        <option value="yes" selected>📞 Yes (Dual SIM)</option>
                        <option value="no">📞 No (Single SIM)</option>
                    </select>
                </div>

                <div class="form-group">
                    <label for="brand">🏷️ Phone Brand</label>
                    <select id="brand" required>
                        <optgroup label="🌟 Premium Brands">
                            <option value="apple">🍎 Apple (iPhone)</option>
                            <option value="samsung">📱 Samsung (Galaxy S/Note)</option>
                        </optgroup>
                        <optgroup label="⭐ High-End Brands">
                            <option value="oneplus">1️⃣ OnePlus</option>
                            <option value="google">🔍 Google Pixel</option>
                            <option value="oppo">🅾️ Oppo</option>
                            <option value="vivo">Ⓥ Vivo</option>
                        </optgroup>
                        <optgroup label="✨ Mid-Range Brands">
                            <option value="xiaomi" selected>⚡ Xiaomi (Mi/Redmi)</option>
                            <option value="realme">🔶 Realme</option>
                            <option value="motorola">Ⓜ️ Motorola</option>
                            <option value="nothing">⭕ Nothing Phone</option>
                        </optgroup>
                        <optgroup label="💰 Budget Brands">
                            <option value="poco">🔥 Poco</option>
                            <option value="infinix">♾️ Infinix</option>
                            <option value="tecno">🔷 Tecno</option>
                            <option value="lava">🌋 Lava</option>
                        </optgroup>
                        <optgroup label="📦 Other Brands">
                            <option value="others">🏢 Others</option>
                        </optgroup>
                    </select>
                </div>

                <button type="submit" class="btn">🔮 Predict Resale Price</button>
            </form>

            <div class="result" id="result">
                <h3>Predicted Resale Price</h3>
                <div class="price" id="predicted_price">₹15,000</div>
                <div class="category" id="price_category">Good Resale Value</div>
            </div>

            <div class="stats">
                <p><strong>Model Accuracy:</strong> {test_acc*100:.1f}%</p>
                <p><strong>Training Samples:</strong> {len(X_train)}</p>
                <p><strong>Algorithm:</strong> Random Forest (AI/ML)</p>
            </div>
        </div>

        <div class="footer">
            <p>By: Jiten Hudda </p>
            <p>AI with Python </p>
        </div>
    </div>

    <script>
        document.getElementById('priceForm').addEventListener('submit', function(e) {{
            e.preventDefault();

            const originalPrice = parseInt(document.getElementById('original_price').value);
            const age = parseFloat(document.getElementById('product_age').value);
            const condition = parseInt(document.getElementById('condition').value);
            const battery = parseInt(document.getElementById('battery').value);
            const storage = parseInt(document.getElementById('storage').value);
            const network = document.getElementById('network').value;
            const dualSim = document.getElementById('dual_sim').value;
            const brand = document.getElementById('brand').value;

            let depreciationRate = 0;

            // Age depreciation (15% per year)
            depreciationRate += age * 0.15;

            // Condition factor
            const conditionFactors = {{1: 0.50, 2: 0.35, 3: 0.20, 4: 0.10, 5: 0.05}};
            depreciationRate += conditionFactors[condition];

            // Battery bonus/penalty
            if (battery >= 6000) depreciationRate -= 0.04;
            else if (battery >= 5000) depreciationRate -= 0.03;
            else if (battery <= 3000) depreciationRate += 0.02;

            // Storage bonus
            if (storage >= 512) depreciationRate -= 0.06;
            else if (storage >= 256) depreciationRate -= 0.05;
            else if (storage >= 128) depreciationRate -= 0.02;

            // Network bonus
            if (network === '5g') depreciationRate -= 0.04;

            // Dual SIM bonus
            if (dualSim === 'yes') depreciationRate -= 0.02;

            // Brand factor (detailed categories)
            const brandFactors = {{
                'apple': -0.08,      // Apple retains value best
                'samsung': -0.06,    // Samsung premium
                'oneplus': -0.03,    // High-end brands
                'google': -0.03,
                'oppo': -0.02,
                'vivo': -0.02,
                'xiaomi': 0,         // Mid-range baseline
                'realme': 0.02,
                'motorola': 0.02,
                'nothing': 0,
                'poco': 0.05,        // Budget brands
                'infinix': 0.08,
                'tecno': 0.08,
                'lava': 0.10,
                'others': 0.06
            }};

            depreciationRate += brandFactors[brand];

            // Cap depreciation between 10% and 75%
            depreciationRate = Math.max(0.10, Math.min(depreciationRate, 0.75));

            const predictedPrice = Math.round(originalPrice * (1 - depreciationRate));

            let category;
            const percentRetained = (predictedPrice / originalPrice) * 100;

            if (percentRetained >= 80) category = "🌟 Excellent Resale Value";
            else if (percentRetained >= 65) category = "⭐ Very Good Resale Value";
            else if (percentRetained >= 50) category = "✨ Good Resale Value";
            else if (percentRetained >= 35) category = "💫 Fair Resale Value";
            else category = "⚠️ Low Resale Value";

            document.getElementById('predicted_price').textContent = '₹' + predictedPrice.toLocaleString('en-IN');
            document.getElementById('price_category').textContent = category;

            const resultDiv = document.getElementById('result');
            resultDiv.classList.add('show');
            resultDiv.scrollIntoView({{ behavior: 'smooth', block: 'nearest' }});
        }});
    </script>
</body>
</html>
"""

display(HTML(html_code))

print("\n" + "=" * 80)
print("✅ ENHANCED INTERFACE LOADED!")
print("=" * 80)
print("\n🎨 New Features:")
print("   ✅ Better colors and visibility")
print("   ✅ Clear emoji icons for each field")
print("   ✅ Detailed brand categories:")
print("      • Premium: Apple, Samsung")
print("      • High-End: OnePlus, Google, Oppo, Vivo")
print("      • Mid-Range: Xiaomi, Realme, Motorola, Nothing")
print("      • Budget: Poco, Infinix, Tecno, Lava")
print("      • Others category")
print("   ✅ Smooth animations and hover effects")
print("   ✅ All fields dynamically affect price!")
print("=" * 80)



✅ ENHANCED INTERFACE LOADED!

🎨 New Features:
   ✅ Better colors and visibility
   ✅ Clear emoji icons for each field
   ✅ Detailed brand categories:
      • Premium: Apple, Samsung
      • High-End: OnePlus, Google, Oppo, Vivo
      • Mid-Range: Xiaomi, Realme, Motorola, Nothing
      • Budget: Poco, Infinix, Tecno, Lava
      • Others category
   ✅ Smooth animations and hover effects
   ✅ All fields dynamically affect price!
